# YOLO_ChestXray: Chest X-ray Pathology Detection (Plan 1 Compliant)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wupinlai/YOLO_ChestXray/blob/main/notebooks/YOLO_ChestXray_Colab_Training.ipynb)

This Google Colab notebook implements the complete 5-stage training pipeline, validation checkpoints, comprehensive IoU & error analysis, standardized automated error annotations, 2x2 error galleries, and 10+ page final PDF report generation as defined in `Plan1.md`.

## 1. Environment & GPU Verification

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive & Kaggle Access (Plan 1)

In [ ]:
from google.colab import drive
import os, json

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/YOLO_ChestXray'
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/reports", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/releases", exist_ok=True)

# Read Kaggle token per Plan 1 specification
token_path = '/content/drive/MyDrive/Secrets/kaggle_token.txt'
kaggle_token = None

if os.path.exists(token_path):
    with open(token_path, 'r') as f:
        kaggle_token = f.read().strip()
    print("Loaded Kaggle token from Google Drive Secrets!")
else:
    try:
        from google.colab import userdata
        kaggle_token = userdata.get('KAGGLE_KEY') or userdata.get('KAGGLE_API_TOKEN')
        print("Loaded Kaggle token from Colab User Secrets!")
    except Exception:
        pass

if kaggle_token:
    os.environ['KAGGLE_KEY'] = kaggle_token
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
        json.dump({'key': kaggle_token}, f)
    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
    print("Kaggle API configured successfully.")
else:
    print("Please provide kaggle_token.txt in Google Drive or set Colab Secret KAGGLE_KEY.")

## 3. Clone Repository & Setup Dependencies

In [ ]:
%cd /content
!git clone https://github.com/wupinlai/YOLO_ChestXray.git
%cd /content/YOLO_ChestXray

# Clone YOLOv7 submodule / official base
!git clone https://github.com/WongKinYiu/yolov7.git yolov7_repo
!cp -r yolov7_repo/* ./ || true
!pip install -r requirements.txt

# Download official pretrained YOLOv7 weights
!wget -nc https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt

## 4. Download Kaggle Dataset & Integrity Verification

In [ ]:
!mkdir -p datasets/chestxray8
!kaggle datasets download -d spritan1/yolo-annotated-chestxray-8-object-detection -p datasets/
!unzip -q -o datasets/yolo-annotated-chestxray-8-object-detection.zip -d datasets/chestxray8/

# Verify integrity
train_imgs = len(os.listdir('datasets/chestxray8/train/images')) if os.path.exists('datasets/chestxray8/train/images') else 0
val_imgs = len(os.listdir('datasets/chestxray8/val/images')) if os.path.exists('datasets/chestxray8/val/images') else 0
print(f"Dataset verified: {train_imgs} training images, {val_imgs} validation images.")

## 5. Dataset Configuration (`configs/chestxray.yaml`)

In [ ]:
import yaml

config = {
    'train': '/content/YOLO_ChestXray/datasets/chestxray8/train/images',
    'val': '/content/YOLO_ChestXray/datasets/chestxray8/val/images',
    'nc': 14,
    'names': [
        'Atelectasis',
        'Cardiomegaly',
        'Effusion',
        'Infiltration',
        'Mass',
        'Nodule',
        'Pneumonia',
        'Pneumothorax',
        'Consolidation',
        'Edema',
        'Emphysema',
        'Fibrosis',
        'Pleural_Thickening',
        'Hernia'
    ]
}

os.makedirs('configs', exist_ok=True)
with open('configs/chestxray.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Created configs/chestxray.yaml successfully:")
!cat configs/chestxray.yaml

## 6. Pre-training: Select Random Baseline Validation Sample

In [ ]:
!python scripts/inference.py --action sample --val-dir datasets/chestxray8/val/images --sample-output-dir reports/final_inference
!cp reports/final_inference/sample_image.jpg {DRIVE_DIR}/reports/ || true

## 7. Stage-wise Training Pipeline (5 Stages, 10 Epochs / Stage)

In [ ]:
# Stage 1: Epochs 1 ~ 10
!python scripts/train.py --weights yolov7.pt --stage 1 --epochs 10 --batch-size 16 --name stage_1
!cp checkpoints/checkpoint_10.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

In [ ]:
# Stage 2: Epochs 11 ~ 20 (Resume from checkpoint_10.pt)
!python scripts/train.py --weights checkpoints/checkpoint_10.pt --stage 2 --epochs 20 --batch-size 16 --name stage_2
!cp checkpoints/checkpoint_20.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

In [ ]:
# Stage 3: Epochs 21 ~ 30
!python scripts/train.py --weights checkpoints/checkpoint_20.pt --stage 3 --epochs 30 --batch-size 16 --name stage_3
!cp checkpoints/checkpoint_30.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

In [ ]:
# Stage 4: Epochs 31 ~ 40
!python scripts/train.py --weights checkpoints/checkpoint_30.pt --stage 4 --epochs 40 --batch-size 16 --name stage_4
!cp checkpoints/checkpoint_40.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

In [ ]:
# Stage 5: Epochs 41 ~ 50
!python scripts/train.py --weights checkpoints/checkpoint_40.pt --stage 5 --epochs 50 --batch-size 16 --name stage_5
!cp checkpoints/checkpoint_50.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

## 8. Validation on Best Model Checkpoint

In [ ]:
!python scripts/validate.py --weights checkpoints/best_model.pt --data configs/chestxray.yaml --name val_results
!cp reports/*.png {DRIVE_DIR}/reports/ || true
!cp reports/*.csv {DRIVE_DIR}/reports/ || true

## 9. Comprehensive Plan 1 Post-Processing & Error Diagnostics
Executes:
- 4 Comparison plots (`map50_comparison.png`, `map50_95_comparison.png`, `precision_recall_comparison.png`, `loss_comparison.png`)
- Full statistical IoU, confidence, and class distribution CSVs & plots
- Standardized automated error annotations (FP, FN, MC, Low IoU)
- 2x2 Error Galleries (`error_gallery.jpg`, `worst_iou_cases.jpg`, `top5_detections.jpg`)
- Final inference verification (`reports/final_inference/inference_result.jpg`)
- 10+ page `final_report.pdf`

In [ ]:
!python scripts/run_plan1_pipeline.py

# Sync all generated reports to Google Drive
!cp -r reports/* {DRIVE_DIR}/reports/
print("All Plan 1 deliverables synced to Google Drive!")

## 10. Display Visual Results & Report Summary

In [ ]:
from IPython.display import Image, display
import glob

print("--- Final Inference Result (Side-by-Side with Ground Truth) ---")
if os.path.exists('reports/final_inference/inference_result.jpg'):
    display(Image(filename='reports/final_inference/inference_result.jpg', width=600))

print("--- Error Gallery (2x2 Grid) ---")
if os.path.exists('reports/error_gallery.jpg'):
    display(Image(filename='reports/error_gallery.jpg', width=700))

print("--- Training Progress Comparison ---")
if os.path.exists('reports/map50_comparison.png'):
    display(Image(filename='reports/map50_comparison.png', width=600))

## 11. Package Release Assets for GitHub Release

In [ ]:
!zip -r {DRIVE_DIR}/releases/Release_50_Plan1_assets.zip \
  checkpoints/checkpoint_50.pt \
  checkpoints/best_model.pt \
  reports/final_report.pdf \
  reports/*.png \
  reports/*.csv \
  reports/error_gallery.jpg

print(f"Plan 1 Release bundle created at {DRIVE_DIR}/releases/Release_50_Plan1_assets.zip")